# Scaling Laws, Data Deduplication (MinHash), DoReMi, and Model Collapse

### Historical Lineage & Theoretical Foundations
- **Chinchilla Optimal Scaling** (*Hoffmann et al., 2022; Kaplan et al., 2020*): Overturned under-trained scaling regimes by proving parameter count $N$ and token count $D$ should scale symmetrically ($D^* \approx 20 N^*$) for compute-optimal pre-training.
- **MinHash Deduplication** (*Broder, 1997; Abbas et al., 2023 SemDeDup*): Solves $O(N^2)$ document comparison bottlenecks by mapping text n-grams into $K$ minimal hash signatures where collision probability exactly equals Jaccard similarity.
- **DoReMi Domain Reweighting** (*Xie et al., 2023*): Dynamically optimizes multi-domain data mixtures using excess loss slack $r_k = \ell_k(\theta) - \ell_k(\theta_\text{ref})$ via exponentiated gradient descent.
- **Recursive Model Collapse** (*Shumailov et al., 2024 Nature*): Proves that recursive training on unfiltered synthetic generations causes distribution tails to vanish and variance to decay exponentially to zero.

> [!WARNING]
> **Decision Boundary (When NOT to Train on Pure Synthetic Data):** Never train generative foundations purely on model-generated outputs without grounded real data or formal deterministic verifiers; unanchored recursive training rapidly collapses lexical and conceptual entropy within 5 generations. Always preserve a $\ge 20\%$ verified real data anchor.

**Topic Connections:** `🔗 Upstream`: [Zeroth-Order MeZO & Synthetic Gradients](06_zeroth_order_mezo_synthetic_gradients_and_hsic_bottleneck.ipynb) | `🔗 Downstream`: [Meta-Parameter Sensitivity & Effective LR](08_meta_parameter_sensitivity_fanova_sobol_meta_landscapes_and_effective_lr.ipynb)


## 2. Mathematical Foundations & Governing Equations

1. **Chinchilla Parametric Scaling Law (Hoffmann et al., 2022):** With $C \approx 6 N D$:
$$L(N, D) = E + \frac{A}{N^\alpha} + \frac{B}{D^\beta} \implies N^* \propto C^{\frac{\beta}{\alpha+\beta}}, \quad D^* \propto C^{\frac{\alpha}{\alpha+\beta}}$$
2. **DoReMi Exponentiated Gradient Domain Reweighting (Xie et al., 2023):**
$$r_k = \ell_k(\theta) - \ell_k(\theta_\text{ref}), \quad w_{t+1} = \text{softmax}\big(\log w_t + \eta r\big)$$
3. **MinHash Jaccard Similarity (Broder, 1997):** Over $K$ hash functions:
$$\hat{J}(S_i, S_j) = \frac{1}{K} \sum_{k=1}^K \mathbb{I}\big[M_{i, k} = M_{j, k}\big]$$
4. **Model Collapse Accumulation Cure (Shumailov et al., 2024):** With real data fraction $\lambda$:
$$\sigma_\text{cured}^2 = (1 - \lambda) \sigma_\text{synth}^2 + \lambda \sigma_\text{real}^2$$
5. **Asymptotic Complexity:** Chinchilla $\mathcal{O}(1)$; DoReMi $\mathcal{O}(K)$; MinHash $\mathcal{O}(B^2 K)$; Variance Cure $\mathcal{O}(1)$.


In [ ]:
import sys
from pathlib import Path
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F

repo_root = Path.cwd().resolve()
while not (repo_root / "core").exists() and repo_root.parent != repo_root:
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
import core
core.setup_env(seed=42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ckpt_path = repo_root / "weights/04_training_methods_and_optimisers/03_training_dynamics_self_optimising_non_backprop/07_scaling_laws_chinchilla_data_dedup_doremi_and_model_collapse.safetensors"
weights, test_in, test_out, meta = core.parity.load_reference_checkpoint(ckpt_path, device=device)
print(f"Device: {device} | Checkpoint: {ckpt_path.name} | Loaded {len(weights)} tensors")


In [ ]:
class LinearAnchor(nn.Module):
    def __init__(self, in_features: int = 8, out_features: int = 4):
        super().__init__()
        self.fc = nn.Linear(in_features, out_features)
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc(x)

def compute_chinchilla_loss_and_opt(n: torch.Tensor, d: torch.Tensor, e: float = 1.69, a: float = 40.0, b: float = 50.0, alpha: float = 0.34, beta: float = 0.28, g_n: float = 0.6) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    loss = e + a * (n ** (-alpha)) + b * (d ** (-beta))
    c = 6.0 * n * d
    n_opt = g_n * (c ** (beta / (alpha + beta)))
    d_opt = (1.0 / (6.0 * g_n)) * (c ** (alpha / (alpha + beta)))
    return loss, n_opt, d_opt

def doremi_update_step(w: torch.Tensor, l_model: torch.Tensor, l_ref: torch.Tensor, eta: float = 1.0) -> tuple[torch.Tensor, torch.Tensor]:
    slack = l_model - l_ref
    return slack, F.softmax(torch.log(w) + eta * slack, dim=-1)

def minhash_pairwise_jaccard(sig: torch.Tensor) -> torch.Tensor:
    return torch.mean((sig.unsqueeze(1) == sig.unsqueeze(0)).float(), dim=-1)

def model_collapse_variance_cure(v_synth: torch.Tensor, v_real: torch.Tensor, lam: float = 0.25) -> torch.Tensor:
    return (1.0 - lam) * v_synth + lam * v_real
print("Scaling laws, deduplication & mixture routines defined.")


In [ ]:
anchor = LinearAnchor(8, 4).to(device)
anchor.fc.weight.data.copy_(weights['anchor_w'].T)
anchor.fc.bias.data.copy_(weights['anchor_b'])

loss, n_opt, d_opt = compute_chinchilla_loss_and_opt(weights['n_val'], weights['d_val'])
err_chinchilla = max(float(torch.abs(loss - weights['golden_chinchilla_loss']).item()),
                     float(torch.abs(n_opt - weights['golden_n_opt']).item()),
                     float(torch.abs(d_opt - weights['golden_d_opt']).item()))

excess_l, w_next = doremi_update_step(weights['w_doremi_init'], weights['losses_model'], weights['losses_ref'])
err_doremi = max(float(torch.max(torch.abs(excess_l - weights['golden_excess_loss'])).item()),
                 float(torch.max(torch.abs(w_next - weights['golden_w_doremi_next'])).item()))

jaccard_m = minhash_pairwise_jaccard(weights['signatures'])
err_minhash = float(torch.max(torch.abs(jaccard_m - weights['golden_jaccard_matrix'])).item())

cured_v = model_collapse_variance_cure(weights['var_synth'], weights['var_real'])
err_collapse = float(torch.abs(cured_v - weights['golden_cured_var']).item())

anchor_out = anchor(weights['anchor_x'])
err_anchor = float(torch.max(torch.abs(anchor_out - weights['golden_anchor_out'])).item())

for name, err in [('Chinchilla', err_chinchilla), ('DoReMi', err_doremi), ('MinHash', err_minhash), ('Collapse', err_collapse), ('Anchor', err_anchor)]:
    assert err < 1e-5, f'{name} invariant failed: {err:.2e}'
print(f'All 5 invariants verified! Chinchilla: {err_chinchilla:.2e}, DoReMi: {err_doremi:.2e}, MinHash: {err_minhash:.2e}, Collapse: {err_collapse:.2e}, Anchor: {err_anchor:.2e}')


In [ ]:
track_fns = {
    'Linear Anchor': lambda: anchor(weights['anchor_x']),
    'Chinchilla Allocation': lambda: compute_chinchilla_loss_and_opt(weights['n_val'], weights['d_val']),
    'DoReMi Reweighting': lambda: doremi_update_step(weights['w_doremi_init'], weights['losses_model'], weights['losses_ref']),
    'MinHash Pairwise Jaccard': lambda: minhash_pairwise_jaccard(weights['signatures']),
    'Model Collapse Mixture': lambda: model_collapse_variance_cure(weights['var_synth'], weights['var_real']),
}
bench_results = core.benchmark.compare_tracks(track_fns, warmup=3, iters=15, batch_size=4)
print(core.benchmark.format_benchmark_table(bench_results))


In [ ]:
# Pathological Anti-Pattern: Unanchored recursive synthetic training causes exponential variance collapse
# Simulating 10 generations of recursive Gaussian generation vs. real-data accumulation
rng = np.random.RandomState(42)
num_samples = 200
var_unanchored = 1.0
var_anchored = 1.0
lam_real = 0.25 # 25% verified real data mixture anchor

for gen in range(10):
    # Synthetic sample generation
    samples_unanchored = rng.normal(0.0, np.sqrt(var_unanchored), num_samples)
    var_unanchored = float(np.var(samples_unanchored)) * 0.85 # Iterative variance attenuation
    # Anchored generation
    samples_anchored = rng.normal(0.0, np.sqrt(var_anchored), num_samples)
    var_synth = float(np.var(samples_anchored)) * 0.85
    var_anchored = (1.0 - lam_real) * var_synth + lam_real * 1.0

print(f'[FAILURE DEMONSTRATED] Unanchored synthetic training variance collapsed to: {var_unanchored:.4f}!')
print(f'[CORRECT] Real-data anchored training preserved statistical variance: {var_anchored:.4f}!')
assert var_unanchored < 0.25 and var_anchored > 0.65, 'Model collapse assertion failed.'
print('[REMEDY] Always accumulate >= 20% authentic real data or enforce strict execution verifiers during synthetic training.')


## 8. Algorithmic Takeaways & Cross-Domain Isomorphisms

- **Takeaways**: Chinchilla scaling mandates symmetric parameter and token growth; MinHash enables sub-quadratic deduplication; DoReMi dynamically balances multi-domain mixtures; real-data accumulation cures recursive model collapse.

| Scientific Domain | Chinchilla Scaling | MinHash Deduplication | DoReMi Mixture Tuning |
| :--- | :--- | :--- | :--- |
| **Large-Scale Pre-Training** | Symmetric parameter/token scaling ($D \approx 20N$) | Eliminates duplicate web documents at scale | Dynamic balance between web, code, math, and papers |
| **Genomic Sequence Alignment** | Model capacity vs sequencing read depth | K-mer hash collisions for bacterial strains | Balancing species representations across phylogenies |
| **Financial Algorithmic Trading** | Model capacity vs tick-level market history | Deduplicating quote spam and phantom orders | Dynamic capital reweighting across volatile asset sectors |

---
**Navigation:** [← Zeroth-Order MeZO & Synthetic Gradients](06_zeroth_order_mezo_synthetic_gradients_and_hsic_bottleneck.ipynb) | [↑ Master Index](../../README.md) | [Meta-Parameter Sensitivity & Effective LR →](08_meta_parameter_sensitivity_fanova_sobol_meta_landscapes_and_effective_lr.ipynb)
